# OceanBase Quickstart

このNotebookでは、GitHub Codespaces上で起動しているOceanBase CEの`users`テーブルを、PyMySQLとpandasで確認・分析します。

## 事前準備

リポジトリのルートで、Dashboardと同じDB接続用の環境変数を設定してからJupyterLabを起動してください。接続情報はNotebookやソースコードに直接書き込みません。

```bash
export OCEANBASE_HOST=127.0.0.1
export OCEANBASE_PORT=2881
export OCEANBASE_USER='root@sys'
export OCEANBASE_PASSWORD=''
export OCEANBASE_DATABASE=test

uv run jupyter lab --ip=0.0.0.0 --port=8888 --no-browser
```

CodespacesのPortsタブで`8888`を開き、このNotebookを表示します。

## 実行順

セルを上から順番に実行してください。

1. 接続セルで環境変数を読み込み、OceanBaseへ接続します。
2. `SELECT VERSION()`でサーバーバージョンを確認します。
3. `users`をSQLで取得し、pandas DataFrameへ読み込みます。
4. DataFrameの内容とレコード件数を表示します。
5. ユーザーごとの件数をmatplotlibで棒グラフにします。
6. 最後のセルでDB接続を閉じます。

途中で接続エラーが発生した場合は、OceanBaseコンテナが起動していること、環境変数の値、Codespaces内から`127.0.0.1:2881`へ接続できることを確認してください。

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import pymysql
from IPython.display import display
from pymysql.cursors import DictCursor

required_variables = [
    "OCEANBASE_HOST",
    "OCEANBASE_PORT",
    "OCEANBASE_USER",
    "OCEANBASE_PASSWORD",
    "OCEANBASE_DATABASE",
]
missing_variables = [name for name in required_variables if name not in os.environ]
if missing_variables:
    raise RuntimeError(f"環境変数が未設定です: {', '.join(missing_variables)}")

connection_config = {
    "host": os.environ["OCEANBASE_HOST"],
    "port": int(os.environ["OCEANBASE_PORT"]),
    "user": os.environ["OCEANBASE_USER"],
    "password": os.environ["OCEANBASE_PASSWORD"],
    "database": os.environ["OCEANBASE_DATABASE"],
    "cursorclass": DictCursor,
    "connect_timeout": 5,
}

connection = pymysql.connect(**connection_config)
print("OceanBaseへの接続に成功しました")

## サーバーバージョン

In [ ]:
with connection.cursor() as cursor:
    cursor.execute("SELECT VERSION() AS version")
    version = cursor.fetchone()["version"]

print(version)

## usersテーブルの読み込み

In [ ]:
with connection.cursor() as cursor:
    cursor.execute("SELECT id, name, created_at FROM users ORDER BY id")
    users = cursor.fetchall()

users_df = pd.DataFrame(users)
users_df

## DataFrameの表示と簡単な集計

In [ ]:
display(users_df)

summary = pd.DataFrame({
    "metric": ["users"],
    "count": [len(users_df)],
})
summary

## usersの可視化

## dbt martの読み込み

`dbt run`が成功した後に実行すると、dbtが作成した`user_summary`テーブルをJupyterから読み込めます。dbt-mysqlとOceanBaseの互換性エラーが発生している場合、このセルは実行できません。

In [ ]:
with connection.cursor() as cursor:
    cursor.execute(
        "SELECT name, user_count, first_created_at, last_created_at "
        "FROM user_summary ORDER BY user_count DESC, name"
    )
    mart_df = pd.DataFrame(cursor.fetchall())

mart_df

In [ ]:
ax = users_df.set_index("name").assign(count=1)["count"].plot(
    kind="bar",
    title="Migrated Users",
    color="#087f73",
    legend=False,
)
ax.set_xlabel("Name")
ax.set_ylabel("Records")
plt.tight_layout()
plt.show()

In [ ]:
connection.close()
print("接続を閉じました")